# 06 - Fault category classifier: training and cross-validation

Trains the classifier in `.spec/features/fault-categories-and-report-page.md`: a fine-tuned
`all-MiniLM-L6-v2` that sorts a report into one of ten categories. It replaces the
two-class leak model of notebooks 04 and 05 and reuses its recipe.

**Data.** [Civic and Municipality Complaint System dataset](https://www.kaggle.com/datasets/wajahattaj/civic-and-municipality-complaint-system-dataset)
(MIT). Download `municipal_training_set_1100.csv` into `ml-research/data/report-classifier/`.
`ml_research.report_category_data` reduces its 1,100 rows to one row per distinct sentence.

**Evaluation** is 5-fold cross-validation over distinct sentences, stratified by category,
so a sentence is never on both sides of a split. Each candidate is chosen on the mean of
three training seeds. The scores are optimistic about the chosen settings, and measure only
US street complaints, not what estate residents write.

A report is *assigned* a category only when its top probability is at least 0.80; below that
it goes to a Manager. Targets at 0.80: precision >= 0.95 for every category except `OTHER`
(a false route emails a contractor for nothing) and macro F1 >= 0.85 over all ten.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline

from ml_research.report_category_data import CATEGORIES, prepare
from ml_research.water_leak_training import device, predict_logits, train_model

DATA = Path("../data/report-classifier/municipal_training_set_1100.csv")
ARTIFACTS = Path("../artifacts/fault-category")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

THRESHOLD = 0.80
TARGET_PRECISION = 0.95
TARGET_MACRO_F1 = 0.85
NEVER_ROUTED = {"OTHER"}
FOLDS = 5
SEED = 42
EXTRA_SEEDS = [7, 123]
K = len(CATEGORIES)

device

## Data

In [ ]:
sentences = prepare(DATA)
texts = sentences["sentence"].tolist()
labels = sentences["target"].to_numpy()

print(len(sentences), "sentences")
print(sentences["label"].value_counts().to_string())
folds = list(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(texts, labels))
print("held out per fold:", [len(test) for _, test in folds])

## Training

The plain PyTorch loop from notebook 04 with a ten-class head. Categories range from 22 to 54
sentences, so the loss is weighted by the square root of balanced class weights, as the leak
model was.

In [ ]:
def cross_validate(epochs, learning_rate, class_weight, seed=SEED):
    held_out = np.zeros((len(texts), K), dtype=np.float32)
    for train, test in folds:
        model = train_model(
            [texts[i] for i in train], labels[train], epochs, learning_rate, class_weight, seed,
            num_labels=K,
        )
        held_out[test] = predict_logits(model, [texts[i] for i in test])
        del model
    return held_out

## Calibration and metrics

One temperature is fitted on the pooled held-out logits. `per_class` scores a threshold as
routing uses it: a report is assigned its top category only if that category's probability is
at least the threshold. Recall is the share of a category's reports assigned to it
automatically. ECE uses the top-label confidence.

In [ ]:
def fit_temperature(logits, targets):
    logits = torch.tensor(logits, dtype=torch.float64)
    targets = torch.tensor(targets)
    log_temperature = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=200)

    def closure():
        optimizer.zero_grad()
        loss = torch.nn.functional.cross_entropy(logits / log_temperature.exp(), targets)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(log_temperature.detach().exp())


def probabilities(logits, temperature=1.0):
    return torch.softmax(torch.tensor(logits, dtype=torch.float64) / temperature, dim=1).numpy()


def per_class(probability, targets, threshold=THRESHOLD):
    top = probability.argmax(axis=1)
    assigned = probability.max(axis=1) >= threshold
    rows = []
    for index, name in enumerate(CATEGORIES):
        predicted = assigned & (top == index)
        actual = targets == index
        tp = int((predicted & actual).sum())
        fp = int((predicted & ~actual).sum())
        fn = int((actual & ~predicted).sum())
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-9)
        rows.append({"category": name, "precision": precision, "recall": recall, "f1": f1,
                     "tp": tp, "fp": fp, "fn": fn, "support": int(actual.sum())})
    return pd.DataFrame(rows).set_index("category")


def calibration_error(probability, targets, bins=10):
    confidence = probability.max(axis=1)
    correct = probability.argmax(axis=1) == targets
    edges = np.linspace(0, 1, bins + 1)
    error = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        members = (confidence > low) & (confidence <= high)
        if members.any():
            error += members.mean() * abs(correct[members].mean() - confidence[members].mean())
    return error


def routed_precision_floor(table):
    return table.drop(index=list(NEVER_ROUTED))["precision"].min()

## Iteration log

Each row is one cross-validation run, seed 42. The TF-IDF row is a reference on the same folds
with its own temperature; it is not a candidate for shipping.

In [ ]:
CONFIGS = [
    {"epochs": 5, "learning_rate": 1e-4, "class_weight": "sqrt"},
    {"epochs": 8, "learning_rate": 1e-4, "class_weight": "sqrt"},
    {"epochs": 12, "learning_rate": 1e-4, "class_weight": "sqrt"},
    {"epochs": 8, "learning_rate": 5e-5, "class_weight": "sqrt"},
    {"epochs": 12, "learning_rate": 5e-5, "class_weight": "sqrt"},
    {"epochs": 8, "learning_rate": 2e-4, "class_weight": "sqrt"},
]


def summarise(name, logits):
    temperature = fit_temperature(logits, labels)
    probability = probabilities(logits, temperature)
    table = per_class(probability, labels)
    assigned = probability.max(axis=1) >= THRESHOLD
    return {
        "run": name,
        "temperature": round(temperature, 3),
        "macro_f1@0.80": round(table["f1"].mean(), 3),
        "min_routed_precision@0.80": round(routed_precision_floor(table), 3),
        "coverage@0.80": round(assigned.mean(), 3),
        "accuracy@argmax": round((probability.argmax(axis=1) == labels).mean(), 3),
        "wrong_assigned@0.80": int(table["fp"].sum()),
        "ece": round(calibration_error(probability, labels), 3),
    }


baseline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
    LogisticRegression(max_iter=2000, class_weight="balanced"),
)
baseline_logits = np.zeros((len(texts), K))
for train, test in folds:
    baseline.fit([texts[i] for i in train], labels[train])
    baseline_logits[test] = baseline.decision_function([texts[i] for i in test])

log = [summarise("TF-IDF reference", baseline_logits)]
runs = {}
for config in CONFIGS:
    name = f"MiniLM epochs={config['epochs']} lr={config['learning_rate']:g} weight={config['class_weight']}"
    start = time.time()
    runs[name] = (config, cross_validate(**config))
    row = summarise(name, runs[name][1])
    row["seconds"] = round(time.time() - start)
    log.append(row)
    print(row, flush=True)

iteration_log = pd.DataFrame(log)
iteration_log.to_csv(ARTIFACTS / "iteration-log.csv", index=False)
baseline_table = per_class(probabilities(baseline_logits, fit_temperature(baseline_logits, labels)), labels)
iteration_log

## Stability across seeds

With 22 to 54 sentences per category, one sentence moves a category's precision by two to five
points. The three best runs are repeated with seeds 7 and 123 and the choice is made on the
per-class mean: highest mean macro F1 among runs whose every routed category meets the
precision target on average, otherwise the highest lowest-category precision.

In [ ]:
minilm = iteration_log[iteration_log["run"].str.startswith("MiniLM")]
shortlist = minilm.sort_values(["min_routed_precision@0.80", "macro_f1@0.80"], ascending=False).head(3)

seed_tables = {}
stability = []
for name in shortlist["run"]:
    config, logits = runs[name]
    seed_runs = [logits] + [cross_validate(**config, seed=seed) for seed in EXTRA_SEEDS]
    tables = [per_class(probabilities(l, fit_temperature(l, labels)), labels) for l in seed_runs]
    mean = sum(t[["precision", "recall", "f1"]] for t in tables) / len(tables)
    seed_tables[name] = (seed_runs, tables, mean)
    stability.append({
        "run": name,
        "mean_macro_f1": round(mean["f1"].mean(), 3),
        "min_routed_mean_precision": round(routed_precision_floor(mean), 3),
        "macro_f1_per_seed": [round(float(t["f1"].mean()), 3) for t in tables],
        "wrong_assigned_per_seed": [int(t["fp"].sum()) for t in tables],
        "temperature_per_seed": [round(fit_temperature(l, labels), 3) for l in seed_runs],
    })
    print(stability[-1], flush=True)
stability = pd.DataFrame(stability)
stability.to_csv(ARTIFACTS / "stability.csv", index=False)

meeting = stability[stability["min_routed_mean_precision"] >= TARGET_PRECISION]
ranked = (meeting.sort_values("mean_macro_f1", ascending=False) if not meeting.empty
          else stability.sort_values(["min_routed_mean_precision", "mean_macro_f1"], ascending=False))
chosen_name = ranked.iloc[0]["run"]
chosen_config, chosen_logits = runs[chosen_name]
temperature = fit_temperature(chosen_logits, labels)
probability = probabilities(chosen_logits, temperature)
seed_runs, tables, mean_table = seed_tables[chosen_name]
print(chosen_name, "temperature (seed 42)", round(temperature, 4))
stability

## Per-category result of the chosen run, mean of three seeds

In [ ]:
report = mean_table.round(3).assign(
    support=tables[0]["support"],
    fp_per_seed=[[int(t.loc[c, "fp"]) for t in tables] for c in CATEGORIES],
    tfidf_precision=baseline_table["precision"].round(3),
    tfidf_recall=baseline_table["recall"].round(3),
)
report["meets_precision"] = [
    "n/a" if c in NEVER_ROUTED else bool(report.loc[c, "precision"] >= TARGET_PRECISION) for c in CATEGORIES
]
missing = [c for c in CATEGORIES if c not in NEVER_ROUTED and report.loc[c, "precision"] < TARGET_PRECISION]
print("mean macro F1", round(mean_table["f1"].mean(), 3), "| TF-IDF macro F1", round(baseline_table["f1"].mean(), 3))
print("categories missing precision", TARGET_PRECISION, ":", missing or "none")
report

## Mistakes

What a person should read before trusting the numbers (seed 42).

In [ ]:
pd.set_option("display.max_colwidth", 110, "display.width", 220, "display.max_rows", 120)
scored = sentences.assign(
    predicted=[CATEGORIES[i] for i in probability.argmax(axis=1)],
    p=probability.max(axis=1).round(3),
)[["sentence", "label", "predicted", "p"]]
wrong = scored["label"] != scored["predicted"]
print("Wrong and assigned at 0.80 (a false route)")
display(scored[wrong & (scored["p"] >= THRESHOLD)].sort_values("p", ascending=False))
print("Wrong but below 0.80 (goes to a Manager)")
display(scored[wrong & (scored["p"] < THRESHOLD)].sort_values("p", ascending=False))
confusion = pd.crosstab(scored["label"], scored["predicted"])
confusion

## Precision and coverage by threshold, and calibration

In [ ]:
thresholds = np.round(np.arange(0.40, 0.96, 0.05), 2)
curve = []
for t in thresholds:
    assigned = probability.max(axis=1) >= t
    correct = probability.argmax(axis=1) == labels
    curve.append({"threshold": t, "coverage": assigned.mean(),
                  "precision": (correct & assigned).sum() / max(assigned.sum(), 1),
                  "macro_f1": per_class(probability, labels, t)["f1"].mean()})
curve = pd.DataFrame(curve).round(3)
display(curve)

confidence = probability.max(axis=1)
correct = probability.argmax(axis=1) == labels
edges = np.linspace(0, 1, 11)
points = [(confidence[m].mean(), correct[m].mean())
          for m in ((confidence > lo) & (confidence <= hi) for lo, hi in zip(edges[:-1], edges[1:])) if m.any()]
figure, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(curve["threshold"], curve["precision"], marker="o", label="precision of assigned")
axes[0].plot(curve["threshold"], curve["coverage"], marker="o", label="coverage (share assigned)")
axes[0].axvline(THRESHOLD, color="grey", linestyle="--")
axes[0].set_xlabel("threshold")
axes[0].legend()
axes[1].plot([0, 1], [0, 1], color="grey", linestyle="--")
axes[1].plot(*zip(*points), marker="o")
axes[1].set_xlabel("confidence")
axes[1].set_ylabel("accuracy")
axes[1].set_title("Reliability")
figure.tight_layout()

In [ ]:
(ARTIFACTS / "cross-validation.json").write_text(json.dumps({
    "run": chosen_name,
    "config": chosen_config,
    "temperature": temperature,
    "threshold": THRESHOLD,
    "folds": FOLDS,
    "seeds": [SEED, *EXTRA_SEEDS],
    "sentences": len(texts),
    "perClassMean": mean_table.round(4).to_dict(orient="index"),
    "macroF1Mean": float(mean_table["f1"].mean()),
    "macroF1PerSeed": [float(t["f1"].mean()) for t in tables],
    "missingPrecision": missing,
    "ece": calibration_error(probability, labels),
    "coverage": float((probability.max(axis=1) >= THRESHOLD).mean()),
    "tfidf": {"perClass": baseline_table[["precision", "recall", "f1"]].round(4).to_dict(orient="index"),
              "macroF1": float(baseline_table["f1"].mean())},
    "precisionCoverage": curve.to_dict(orient="records"),
}, indent=2))

## Results

Recorded in `.spec/features/fault-categories-and-report-page.md` ("Cross-validation result") and `artifacts/fault-category/README.md`, since outputs are cleared before committing.